# Reset / Cleanup Notebook (Test Environment)

This notebook clears:
- Landing CSV files (optionally per test folder)
- Raw target table (e.g., `raw.lse_equities_prices_test`)
- DQ audit tables (`governance.dq_run`, `governance.dq_check_result`)
- Auto Loader checkpoint + schema tracking directories

⚠️ **WARNING**: This is destructive. Use only in non-production/test environments.

In [0]:
# Widgets

dbutils.widgets.text("config_path", "", "YAML config path")
dbutils.widgets.dropdown(
    "test_folder",
    "all",
    ["all", "good", "few_rows", "required_nulls", "high_null_close", "good_001", "few_rows_001", "required_nulls_001", "high_null_close_001"],
    "Landing subfolder to clear (optional)"
)
dbutils.widgets.dropdown("clear_landing", "true", ["true", "false"], "Clear landing files")
dbutils.widgets.dropdown("clear_raw_table", "true", ["true", "false"], "Clear raw target table")
dbutils.widgets.dropdown("clear_dq_tables", "true", ["true", "false"], "Clear DQ audit tables")
dbutils.widgets.dropdown("clear_checkpoint", "true", ["true", "false"], "Clear streaming checkpoint")
dbutils.widgets.dropdown("clear_schema_loc", "true", ["true", "false"], "Clear Auto Loader schema location")

config_path = dbutils.widgets.get("config_path").strip()
selected = dbutils.widgets.get("test_folder").strip()

clear_landing = dbutils.widgets.get("clear_landing").lower() == "true"
clear_raw_table = dbutils.widgets.get("clear_raw_table").lower() == "true"
clear_dq_tables = dbutils.widgets.get("clear_dq_tables").lower() == "true"
clear_checkpoint = dbutils.widgets.get("clear_checkpoint").lower() == "true"
clear_schema_loc = dbutils.widgets.get("clear_schema_loc").lower() == "true"

if not config_path:
    raise ValueError("config_path is empty. Provide the YAML config path used by the ingestion notebook.")

In [0]:
# Load YAML config

import yaml
cfg_text = dbutils.fs.head(config_path, 100000)
cfg = yaml.safe_load(cfg_text)

landing_path = cfg["landing_path"]
raw_table = cfg["raw_table"]
checkpoint = cfg.get("raw_checkpoint") or cfg.get("checkpoint")
schema_loc = cfg.get("schema_location") or cfg.get("schema_loc")

print("Loaded:")
print(" landing_path:", landing_path)
print(" raw_table   :", raw_table)
print(" checkpoint  :", checkpoint)
print(" schema_loc  :", schema_loc)

Loaded:
 landing_path: abfss://unity-catalog-storage@dbstoragezm5deu7lm74ky.dfs.core.windows.net/7405613464818727/macro_platform/landing/lse/equities_prices_test/
 raw_table   : raw.lse_equities_prices_test
 checkpoint  : abfss://unity-catalog-storage@dbstoragezm5deu7lm74ky.dfs.core.windows.net/7405613464818727/macro_platform/checkpoints/raw/lse/equities_prices_test/
 schema_loc  : abfss://unity-catalog-storage@dbstoragezm5deu7lm74ky.dfs.core.windows.net/7405613464818727/macro_platform/schema/raw/lse/equities_prices_test/


In [0]:
# Stop any active streams to release checkpoint locks

active = spark.streams.active
if active:
    print(f"Stopping {len(active)} active stream(s)...")
    for s in active:
        try:
            print(" - stopping:", s.id)
            s.stop()
        except Exception as e:
            print("   ! failed to stop stream:", e)
else:
    print("No active streams.")

No active streams.


In [0]:
# Helper functions

def _exists_path(p: str) -> bool:
    try:
        dbutils.fs.ls(p)
        return True
    except Exception:
        return False

def rm_path(p: str, recurse: bool = True):
    if not p:
        print("Skip: empty path")
        return
    if _exists_path(p):
        print("Removing:", p)
        dbutils.fs.rm(p, recurse=recurse)
    else:
        print("Not found (skip):", p)

def table_exists(qualified_name: str) -> bool:
    try:
        return spark.catalog.tableExists(qualified_name)
    except Exception:
        # Fallback for some catalogs
        parts = qualified_name.split(".")
        if len(parts) == 2:
            return spark._jsparkSession.catalog().tableExists(parts[0], parts[1])
        return spark._jsparkSession.catalog().tableExists(qualified_name)

In [0]:
# 1) Clear landing files

if clear_landing:
    landing_to_clear = landing_path
    if selected and selected.lower() != "all":
        landing_to_clear = f"{landing_path.rstrip('/')}/{selected}/"

    print("Landing to clear:", landing_to_clear)

    # Safety: only remove within landing_to_clear
    rm_path(landing_to_clear, recurse=True)
    # Recreate base directory so downstream notebooks don't fail on missing paths
    try:
        dbutils.fs.mkdirs(landing_to_clear)
    except Exception:
        pass
else:
    print("Skipping landing cleanup.")

Landing to clear: abfss://unity-catalog-storage@dbstoragezm5deu7lm74ky.dfs.core.windows.net/7405613464818727/macro_platform/landing/lse/equities_prices_test/
Removing: abfss://unity-catalog-storage@dbstoragezm5deu7lm74ky.dfs.core.windows.net/7405613464818727/macro_platform/landing/lse/equities_prices_test/


In [0]:
# 2) Clear raw target table

if clear_raw_table:
    if table_exists(raw_table):
        print("Clearing raw table:", raw_table)
        # TRUNCATE is fastest and keeps schema
        spark.sql(f"TRUNCATE TABLE {raw_table}")
    else:
        print("Raw table not found (skip):", raw_table)
else:
    print("Skipping raw table cleanup.")

Clearing raw table: raw.lse_equities_prices_test


In [0]:
# 3) Clear DQ audit tables

if clear_dq_tables:
    dq_run = "governance.dq_run"
    dq_res = "governance.dq_check_result"

    for t in [dq_run, dq_res]:
        if table_exists(t):
            print("Clearing DQ table:", t)
            spark.sql(f"TRUNCATE TABLE {t}")
        else:
            print("DQ table not found (skip):", t)
else:
    print("Skipping DQ tables cleanup.")

Clearing DQ table: governance.dq_run
Clearing DQ table: governance.dq_check_result


In [0]:
# 4) Clear checkpoint + schema location

if clear_checkpoint:
    print("Clearing checkpoint:", checkpoint)
    rm_path(checkpoint, recurse=True)
else:
    print("Skipping checkpoint cleanup.")

if clear_schema_loc:
    print("Clearing schema location:", schema_loc)
    rm_path(schema_loc, recurse=True)
else:
    print("Skipping schema location cleanup.")

print("✅ Cleanup complete")

Clearing checkpoint: abfss://unity-catalog-storage@dbstoragezm5deu7lm74ky.dfs.core.windows.net/7405613464818727/macro_platform/checkpoints/raw/lse/equities_prices_test/
Removing: abfss://unity-catalog-storage@dbstoragezm5deu7lm74ky.dfs.core.windows.net/7405613464818727/macro_platform/checkpoints/raw/lse/equities_prices_test/
Clearing schema location: abfss://unity-catalog-storage@dbstoragezm5deu7lm74ky.dfs.core.windows.net/7405613464818727/macro_platform/schema/raw/lse/equities_prices_test/
Removing: abfss://unity-catalog-storage@dbstoragezm5deu7lm74ky.dfs.core.windows.net/7405613464818727/macro_platform/schema/raw/lse/equities_prices_test/
✅ Cleanup complete
